In [1]:
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
import torch
import gensim.downloader
import pandas as pd
import spacy
from collections import defaultdict
import random
import json

In [2]:
dataset = load_dataset("wikitext", "wikitext-103-v1", split="train")
#glove = gensim.downloader.load("glove-wiki-gigaword-50")

README.md: 0.00B [00:00, ?B/s]

wikitext-103-v1/test-00000-of-00001.parq(…):   0%|          | 0.00/722k [00:00<?, ?B/s]

wikitext-103-v1/train-00000-of-00002.par(…):   0%|          | 0.00/156M [00:00<?, ?B/s]

wikitext-103-v1/train-00001-of-00002.par(…):   0%|          | 0.00/156M [00:00<?, ?B/s]

wikitext-103-v1/validation-00000-of-0000(…):   0%|          | 0.00/655k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [3]:
df = pd.read_csv("/home/onyxia/work/morph_families.tsv", 
                sep="\t", 
                names=["lemma", "forms", "fam", "transf"])
                
#glove_voc = set(glove.key_to_index.keys())
#print(glove_voc)


In [4]:
content = []
for line in dataset["text"]:
    line = line.strip().lower()
    if line and not line.startswith("=") and len(line.split()) > 4:
        content.append(line)

In [5]:
target = {word for form in df["forms"].str.split(",") for word in form}

In [ ]:
nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")
seen = set()
sentences =  []
for doc in nlp.pipe(content, batch_size=1024, n_process=4):
    for sent in doc.sents:
        if sent.text and sent.text not in seen:
            sentences.append(sent)
            seen.add(sent.text)

In [ ]:
def tag_word(sents):
    n = len(sents)
    if n >= 50:
        return "sampled"
    elif n >= 10:
        return "low_freq"
    return "excluded"

def sample_sents(row):
    if row["tag"] == "sampled":
        return random.sample(row["sents"], 50)
    return row["sents"]


matched = defaultdict(list)
for sent in sentences:
    if len(sent) > 40:
        continue
    tokens = {token.text for token in sent}
    for word in target.intersection(tokens):         
        matched[word].append(sent)

random.seed(42)

sent_df = pd.DataFrame(matched.items(), 
                        columns=["word", "sents"])

sent_df["tag"] = sent_df["sents"].apply(tag_word)
sent_df["sents"] = sent_df.apply(sample_sents, axis="columns")

In [ ]:
with open("sent_df", "w") as f:
    json.dump(sent_df, f)

In [8]:
df.sample()

,lemma,forms,fam,transf
33,nation,"nation,nations,national,international",derivation,noun_to_adj_prefix
